In [1]:
import numpy as np

class GridWorld:
    def __init__(self):
        # 1. Defines the grid and states [cite: 13]
        self.size = 4 # 4x4 grid [cite: 7]
        self.actions = ['U', 'D', 'L', 'R'] # Actions [cite: 11]
        self.goal = (3, 3) # Goal state [cite: 8]
        self.pit = (2, 2) # Pit state [cite: 9]

    def get_next_state_reward(self, state, action):
        # 2. Returns the next state and reward [cite: 14]
        if state == self.goal or state == self.pit:
            return state, 0 
        
        r, c = state
        if action == 'U': r = max(0, r - 1)
        elif action == 'D': r = min(self.size - 1, r + 1)
        elif action == 'L': c = max(0, c - 1)
        elif action == 'R': c = min(self.size - 1, c + 1)
        
        next_state = (r, c)
        
        if next_state == self.goal: reward = 10 # [cite: 8]
        elif next_state == self.pit: reward = -10 # [cite: 9]
        else: reward = -1 # [cite: 10]
        
        return next_state, reward

In [2]:
def value_iteration(env, gamma, theta=0.001):
    # Implement Value Iteration [cite: 16]
    V = np.zeros((env.size, env.size))
    while True:
        delta = 0
        for r in range(env.size):
            for c in range(env.size):
                state = (r, c)
                if state == env.goal or state == env.pit: continue
                v = V[state]
                V[state] = max(env.get_next_state_reward(state, a)[1] + gamma * V[env.get_next_state_reward(state, a)[0]] for a in env.actions)
                delta = max(delta, abs(v - V[state]))
        if delta < theta: break # Stop condition [cite: 17]
    return V

In [3]:
def policy_iteration(env, gamma, theta=0.001):
    V = np.zeros((env.size, env.size))
    policy = { (r, c): 'U' for r in range(env.size) for c in range(env.size) }
    
    while True:
        # 1. Policy Evaluation [cite: 20]
        while True:
            delta = 0
            for r in range(env.size):
                for c in range(env.size):
                    state = (r, c)
                    if state == env.goal or state == env.pit: continue
                    v = V[state]
                    next_s, rwd = env.get_next_state_reward(state, policy[state])
                    V[state] = rwd + gamma * V[next_s]
                    delta = max(delta, abs(v - V[state]))
            if delta < theta: break # [cite: 20]
            
        # 2. Policy Improvement [cite: 21]
        policy_stable = True
        for r in range(env.size):
            for c in range(env.size):
                state = (r, c)
                if state == env.goal or state == env.pit: continue
                old_action = policy[state]
                best_val = -float('inf')
                best_action = old_action
                for a in env.actions:
                    next_s, rwd = env.get_next_state_reward(state, a)
                    val = rwd + gamma * V[next_s]
                    if val > best_val:
                        best_val = val
                        best_action = a
                policy[state] = best_action
                if old_action != best_action: policy_stable = False
        # 3. Stop when policy stable [cite: 22]
        if policy_stable: break 
    return V, policy

In [4]:
# Run Experiment [cite: 23]
env = GridWorld()
for gamma in [0.0, 0.9]: # For gamma 0.0 and 0.9 [cite: 24]
    print(f"\n--- Results for Gamma = {gamma} ---")
    
    # Output Value Iteration value function (4x4) [cite: 25]
    V_vi = value_iteration(env, gamma)
    print("Value Iteration V (4x4):\n", np.round(V_vi, 2))
    
    # Output Policy Iteration value function (4x4) and policy (4x4) [cite: 26]
    V_pi, policy_pi = policy_iteration(env, gamma)
    print("\nPolicy Iteration V (4x4):\n", np.round(V_pi, 2))
    
    print("\nPolicy Iteration Policy (4x4) ['T' indicates Terminal]:")
    for r in range(env.size):
        row = [policy_pi[(r,c)] if (r,c) not in [env.goal, env.pit] else 'T' for c in range(env.size)]
        print(row)


--- Results for Gamma = 0.0 ---
Value Iteration V (4x4):
 [[-1. -1. -1. -1.]
 [-1. -1. -1. -1.]
 [-1. -1.  0. 10.]
 [-1. -1. 10.  0.]]

Policy Iteration V (4x4):
 [[-1. -1. -1. -1.]
 [-1. -1. -1. -1.]
 [-1. -1.  0. 10.]
 [-1. -1. 10.  0.]]

Policy Iteration Policy (4x4) ['T' indicates Terminal]:
['U', 'U', 'U', 'U']
['U', 'U', 'U', 'U']
['U', 'U', 'T', 'D']
['U', 'U', 'R', 'T']

--- Results for Gamma = 0.9 ---
Value Iteration V (4x4):
 [[ 1.81  3.12  4.58  6.2 ]
 [ 3.12  4.58  6.2   8.  ]
 [ 4.58  6.2   0.   10.  ]
 [ 6.2   8.   10.    0.  ]]

Policy Iteration V (4x4):
 [[ 1.81  3.12  4.58  6.2 ]
 [ 3.12  4.58  6.2   8.  ]
 [ 4.58  6.2   0.   10.  ]
 [ 6.2   8.   10.    0.  ]]

Policy Iteration Policy (4x4) ['T' indicates Terminal]:
['D', 'D', 'D', 'D']
['D', 'D', 'R', 'D']
['D', 'D', 'T', 'D']
['R', 'R', 'R', 'T']
